In [1]:
import warnings
import pandas as pd
warnings.filterwarnings("ignore")
from building_models.commons_functions.parsers_commons import ParsersCommons
from building_models.utils.constants import COLUMNS_TO_WORK
from building_models.utils.utils_functions import UtilsFunctions

#### Antioxidant protein dataset preprocessing and metadata generation

This script processes antioxidant protein data from multiple files within a single source, extracts and standardizes labels from sequence identifiers, handles duplicated sequences, and generates a clean dataset along with metadata for downstream classification tasks.

- Overview
    - Task: Antioxidant protein classification dataset preparation
    - Source: AOPxSVM dataset
    - Input: Multiple FASTA files (training and test sets) with embedded labels and metadata Excel file
    - Output: Processed dataset (CSV) and metadata file (JSON)
- Process:
    - Read protein sequences from multiple FASTA files
    - Extract labels from sequence identifiers
    - Standardize label values when needed
    - Concatenate all files into a single dataset
    - Remove identifier column after label extraction
    - Convert labels to integer format
    - Check for duplicated sequences and label consistency
    - Concatenate consistent duplicates with unique sequences
    - Load and organize source metadata
    - Export processed data and metadata

- Auxiliary variables

In [2]:
path_export = "../../processed_dataset"
path_input = "../../raw_dataset"
metadata_file = "../../raw_dataset/raw_data_description.xlsx"
name_task = "antioxidant_classification"
name_source = "AOPxSVM"

- Read doc and labels

In [3]:
df_data_test_2023 = ParsersCommons.read_fasta_doc(f"{path_input}/{name_source}/AOPP.test.2023.fasta")
df_data_test_2023["label"] = df_data_test_2023["id"].str.split("|").str[-1]
df_data_test_2023['label'] = df_data_test_2023['label'].replace('ANOXI', 0)
df_data_test_2023

,id,sequence,label
0,1pos|AntioxidantTest|1,HLLPK,1
1,2pos|AntioxidantTest|1,KEFFP,1
2,3pos|AntioxidantTest|1,KEFFPA,1
3,4pos|AntioxidantTest|1,WPPLSPFRCPR,1
4,5pos|AntioxidantTest|1,IPDWFLNRQ,1
...,...,...,...
145,71neg|ANOXI,WAMTVT,0
146,72neg|ANOXI,FWGCHR,0
147,73neg|ANOXI,WTVECKGGSKI,0
148,74neg|ANOXI,MRF,0


In [4]:
df_data_test = ParsersCommons.read_fasta_doc(f"{path_input}/{name_source}/AOPP.test.fasta")
df_data_test["label"] = df_data_test["id"].str.split("|").str[-1]
df_data_test

,id,sequence,label
0,1pos|AntioxidantTest|1,WV,1
1,2pos|AntioxidantTest|1,LW,1
2,3pos|AntioxidantTest|1,KD,1
3,4pos|AntioxidantTest|1,YP,1
4,5pos|AntioxidantTest|1,IR,1
...,...,...,...
601,299neg|AntioxidantTest|0,RFMPN,0
602,300neg|AntioxidantTest|0,IDFNR,0
603,301neg|AntioxidantTest|0,KHYQQDCYMGNN,0
604,302neg|AntioxidantTest|0,EMLMW,0


In [5]:
df_data_train = ParsersCommons.read_fasta_doc(f"{path_input}/{name_source}/AOPP.train.fasta")
df_data_train["label"] = df_data_train["id"].str.split("|").str[-1]
df_data_train

,id,sequence,label
0,1pos|AntioxidantTrain|1,HK,1
1,2pos|AntioxidantTrain|1,AH,1
2,3pos|AntioxidantTrain|1,HL,1
3,4pos|AntioxidantTrain|1,HH,1
4,5pos|AntioxidantTrain|1,LH,1
...,...,...,...
2411,1204neg|AntioxidantTrain|0,QVECTY,0
2412,1205neg|AntioxidantTrain|0,GGMHCNTWEPT,0
2413,1206neg|AntioxidantTrain|0,CGTMF,0
2414,1207neg|AntioxidantTrain|0,HNFW,0


In [6]:
df_data = pd.concat([df_data_test_2023, df_data_test, df_data_train], ignore_index=True)
df_data = df_data.drop(columns=["id"])
df_data

,sequence,label
0,HLLPK,1
1,KEFFP,1
2,KEFFPA,1
3,WPPLSPFRCPR,1
4,IPDWFLNRQ,1
...,...,...
3167,QVECTY,0
3168,GGMHCNTWEPT,0
3169,CGTMF,0
3170,HNFW,0


In [7]:
df_data["label"] = df_data["label"].astype(int)

- Checking duplicates

In [8]:
df_consistent_duplicates, df_errors, df_unique = ParsersCommons.processing_duplicated(
    df_data, group_seq= "sequence",
    label_col= "label")
df_consistent_duplicates.shape, df_errors.shape, df_unique.shape

((8, 3), (10, 3), (3136, 2))

In [9]:
data_correct = pd.concat([df_consistent_duplicates, df_unique], axis=0, ignore_index=True)
data_correct = data_correct.drop(columns=["n_duplicates"])
data_correct.head()

,sequence,label
0,CLN,0
1,DMY,0
2,DQM,0
3,HCC,0
4,II,0


- Reading metadata

In [10]:
metadata_file = ParsersCommons.read_metadata(metadata_file, name_source=name_source, columns_to_select=COLUMNS_TO_WORK)
metadata_file.head()

,name dataset,name source,type source,static-dynamic,license,reports constant updates,year of publication,last update date,download date,file format,protein format,category dataset,task,obtaining negative dataset,obtaining positive dataset,repository or server,publication,unit of measurement
41,AOPP.test.2023.fasta,AOPxSVM,Dataset,Static,No information,No,2025,2025-05-25,2026-04-07,fasta,Sequence,Enzyme/protein classification,Antioxidant,Previously published model dataset,Previously published model dataset,https://github.com/yashdui/AOPxSVM,https://www.mdpi.com/2304-8158/14/12/2014#app1...,No information
42,AOPP.test.fasta,AOPxSVM,Dataset,Static,No information,No,2025,2025-05-25,2026-04-07,fasta,Sequence,Enzyme/protein classification,Antioxidant,Previously published model dataset,Previously published model dataset,https://github.com/yashdui/AOPxSVM,https://www.mdpi.com/2304-8158/14/12/2014#app1...,No information
43,AOPP.train.fasta,AOPxSVM,Dataset,Static,No information,No,2025,2025-05-25,2026-04-07,fasta,Sequence,Enzyme/protein classification,Antioxidant,Previously published model dataset,Previously published model dataset,https://github.com/yashdui/AOPxSVM,https://www.mdpi.com/2304-8158/14/12/2014#app1...,No information


In [11]:
dict_metadata = ParsersCommons.create_metadata_from_file(metadata_file)
dict_metadata

{'name dataset': 'AOPP.test.2023.fasta;AOPP.test.fasta;AOPP.train.fasta',
 'name source': 'AOPxSVM',
 'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'reports constant updates': 'No',
 'year of publication': 2025,
 'last update date': Timestamp('2025-05-25 00:00:00'),
 'download date': Timestamp('2026-04-07 00:00:00'),
 'file format': 'fasta',
 'protein format': 'Sequence',
 'category dataset': 'Enzyme/protein classification',
 'task': 'Antioxidant',
 'obtaining negative dataset': 'Previously published model dataset',
 'obtaining positive dataset': 'Previously published model dataset',
 'repository or server': 'https://github.com/yashdui/AOPxSVM',
 'publication': 'https://www.mdpi.com/2304-8158/14/12/2014#app1-foods-14-02014',
 'unit of measurement': 'No information',
 'number_of_sources': 3,
 'processing_date': '2026-08-01 19:42:26'}

In [12]:
dict_metadata['number_of_records'] = df_data.shape[0]
dict_metadata['number_of_collected_sequences'] = df_data.shape[0]
dict_metadata['number_of_unique_sequences'] = data_correct.shape[0]
dict_metadata['positive_examples'] = data_correct[data_correct["label"] == 1].shape[0]
dict_metadata['negative_examples'] = data_correct[data_correct["label"] == 0].shape[0]
dict_metadata['number_of_sequences_with_errors'] = df_errors.shape[0]
dict_metadata

{'name dataset': 'AOPP.test.2023.fasta;AOPP.test.fasta;AOPP.train.fasta',
 'name source': 'AOPxSVM',
 'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'reports constant updates': 'No',
 'year of publication': 2025,
 'last update date': Timestamp('2025-05-25 00:00:00'),
 'download date': Timestamp('2026-04-07 00:00:00'),
 'file format': 'fasta',
 'protein format': 'Sequence',
 'category dataset': 'Enzyme/protein classification',
 'task': 'Antioxidant',
 'obtaining negative dataset': 'Previously published model dataset',
 'obtaining positive dataset': 'Previously published model dataset',
 'repository or server': 'https://github.com/yashdui/AOPxSVM',
 'publication': 'https://www.mdpi.com/2304-8158/14/12/2014#app1-foods-14-02014',
 'unit of measurement': 'No information',
 'number_of_sources': 3,
 'processing_date': '2026-08-01 19:42:26',
 'number_of_records': 3172,
 'number_of_collected_sequences': 3172,
 'number_of_unique_sequences': 3144,
 'positive

- Export data

In [13]:
UtilsFunctions.make_directory(f"{path_export}/{name_task}/{name_source}")
UtilsFunctions.export_json(f"{path_export}/{name_task}/{name_source}/metadata_{name_source}.json", dict_metadata)
data_correct.to_csv(f"{path_export}/{name_task}/{name_source}/processed_data.csv", index=False)